# 02 — Training: X-Ray Model (EfficientNet-B4)
هندرّب نموذج الـ X-Ray على Pneumothorax + Pneumonia.

**الاستراتيجية:**
1. Freeze backbone → train head فقط (5 epochs)
2. Unfreeze آخر blocks → fine-tune كل حاجة (10 epochs)

In [ ]:
!git clone https://github.com/YOUR_USERNAME/acute-triage-system.git
%cd acute-triage-system
!pip install -q -r requirements.txt

In [ ]:
import sys; sys.path.insert(0, '.')
import torch, pandas as pd
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

from src.preprocessing import XRayDataset, get_train_transforms, get_val_transforms
from src.models import get_model

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

: 

In [ ]:
# ── Build DataLoaders ──────────────────────────────────────────
df       = pd.read_csv('data/processed/xray_manifest.csv')
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)

train_ds = XRayDataset(train_df, transforms=get_train_transforms())
val_ds   = XRayDataset(val_df,   transforms=get_val_transforms())

train_dl = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} samples | Val: {len(val_ds)} samples')

In [ ]:
# ── Training helpers ───────────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in tqdm(loader, leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(imgs)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(imgs)
    return total_loss / total, correct / total

@torch.no_grad()
def val_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        total_loss += loss.item() * len(imgs)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(imgs)
    return total_loss / total, correct / total

In [ ]:
# ── Phase 1: Train head only (backbone frozen) ─────────────────
model = get_model('xray', num_classes=3, pretrained=True, device=DEVICE)
model.freeze_backbone()

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                  lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=5)

best_val_acc = 0
for epoch in range(5):
    tr_loss, tr_acc = train_epoch(model, train_dl, optimizer, criterion)
    vl_loss, vl_acc = val_epoch(model, val_dl, criterion)
    scheduler.step()
    print(f'Epoch {epoch+1:02d} | '
          f'Train loss {tr_loss:.4f} acc {tr_acc:.3f} | '
          f'Val loss {vl_loss:.4f} acc {vl_acc:.3f}')
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        model.save('models/weights/xray_phase1.pth')

In [ ]:
# ── Phase 2: Fine-tune last 2 blocks ──────────────────────────
model.unfreeze_backbone(last_n_blocks=2)

optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                  lr=5e-5, weight_decay=1e-4)  # Lower LR for fine-tuning!
scheduler = CosineAnnealingLR(optimizer, T_max=10)

best_val_acc = 0
for epoch in range(10):
    tr_loss, tr_acc = train_epoch(model, train_dl, optimizer, criterion)
    vl_loss, vl_acc = val_epoch(model, val_dl, criterion)
    scheduler.step()
    print(f'Epoch {epoch+1:02d} | '
          f'Train loss {tr_loss:.4f} acc {tr_acc:.3f} | '
          f'Val loss {vl_loss:.4f} acc {vl_acc:.3f}')
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        model.save('models/weights/xray_best.pth')
        print(f'  --> New best: {best_val_acc:.3f}')

In [ ]:
# ── Save to Google Drive ───────────────────────────────────────
import shutil
shutil.copy('models/weights/xray_best.pth',
            '/content/drive/MyDrive/acute_triage/xray_best.pth')
print('Saved to Google Drive!')